# SRE On-Call Agent
> AI-powered incident triage, root cause investigation, and post-mortem generation

## Project Introduction

When a production alert fires at 3am, an SRE on-call engineer must:
1. **Triage** — assess severity and plan the investigation
2. **Investigate** — search logs, query metrics, consult runbooks
3. **Write a post-mortem** — root cause, timeline, action items

This project automates that workflow with a **three-agent AI pipeline**:

```
Alert JSON
    │
    ▼
┌─────────────────────────────┐
│  Stage 1: TriageAgent       │  Plan-and-Solve
│  "What should I investigate?"│  → ordered investigation plan
└─────────────┬───────────────┘
              │
              ▼
┌─────────────────────────────┐
│  Stage 2: InvestigationAgent│  ReAct loop
│  log_search / metric_query  │  → root cause hypothesis
│  / runbook_lookup           │
└─────────────┬───────────────┘
              │
              ▼
┌─────────────────────────────┐
│  Stage 3: PostmortemAgent   │  Reflection
│  draft → critique → revise  │  → final RCA report
└─────────────────────────────┘
```

## Author Information
- **GitHub**: [@zjzhou](https://github.com/zjzhou)
- **Date**: 2026-04-21
- **HelloAgents Chapter**: 16 (Graduation Project)

---
## Part 2: Environment Setup

In [1]:
# Install dependencies
!pip install -q openai fastapi uvicorn pyyaml pydantic python-dotenv

In [2]:
import os
import sys
import json
import time
import re
import ast
import glob
import yaml
from pathlib import Path
from typing import Dict, Any, List
from dotenv import load_dotenv
from openai import OpenAI

# Add project root to path so src/ imports work
project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv()
print("✅ Environment loaded")
print(f"   LLM_MODEL_ID: {os.getenv('LLM_MODEL_ID', 'NOT SET')}")
print(f"   LLM_BASE_URL: {os.getenv('LLM_BASE_URL', 'NOT SET')}")
print(f"   LLM_API_KEY:  {'SET' if os.getenv('LLM_API_KEY') else 'NOT SET'}")

✅ Environment loaded
   LLM_MODEL_ID: deepseek-v4-flash
   LLM_BASE_URL: https://api.deepseek.com
   LLM_API_KEY:  SET


---
## Part 3: Tool Definitions

Three tools give the InvestigationAgent eyes into the incident:

| Tool | Input | What it does |
|---|---|---|
| `LogSearchTool` | keyword/regex | Searches incident log entries |
| `MetricQueryTool` | metric name keyword | Returns time-series metric data |
| `RunbookLookupTool` | error pattern | Fetches runbook remediation steps |

In [3]:
from src.tools.log_search_tool import LogSearchTool
from src.tools.metric_query_tool import MetricQueryTool
from src.tools.runbook_tool import RunbookLookupTool

# --- Quick tool smoke test ---
incident_path = project_root / "data" / "incidents" / "db_pool_exhaustion.json"
with open(incident_path, encoding="utf-8") as f:
    sample_incident = json.load(f)

log_tool = LogSearchTool(sample_incident)
metric_tool = MetricQueryTool(sample_incident)
runbook_tool = RunbookLookupTool(
    service="checkout-service",
    runbooks_dir=str(project_root / "data" / "runbooks")
)

print("🔧 LogSearchTool — searching for 'pool':")
print(log_tool.run("pool"))
print()
print("📊 MetricQueryTool — querying 'db_pool':")
print(metric_tool.run("db_pool"))
print()
print("📖 RunbookLookupTool — pattern 'pool exhausted':")
print(runbook_tool.run("pool exhausted"))

🔧 LogSearchTool — searching for 'pool':
Found 4 log entries matching 'pool':
[2024-01-15T14:00:03Z] [WARN    ] DB connection pool: 8/10 connections in use
[2024-01-15T14:01:15Z] [WARN    ] DB connection pool: 10/10 connections in use, queuing requests
[2024-01-15T14:01:45Z] [ERROR   ] DB connection pool exhausted — waiting for available connection (timeout=30s)
[2024-01-15T14:02:01Z] [ERROR   ] DB connection pool exhausted — waiting for available connection (timeout=30s)

📊 MetricQueryTool — querying 'db_pool':
Metrics for checkout-service matching 'db_pool':
  db_pool_active_connections: [14:00: 3 | 14:01: 8 | 14:02: 10 | 14:03: 10]

📖 RunbookLookupTool — pattern 'pool exhausted':
Runbook: checkout-service (v1.2)

Pattern: 'DB pool exhausted' | Severity: P1
Remediation steps:
  1. Identify slow queries: run SHOW PROCESSLIST on the primary DB; look for queries >5s
  2. Run EXPLAIN on the slowest query to identify full table scans or missing indexes
  3. Temporary mitigation: increase p

---
## Part 4: Agent Construction

Three agents implementing different paradigms from the Hello-Agents curriculum:

- **TriageAgent** — Plan-and-Solve (Chapter 4): LLM creates a structured investigation plan
- **InvestigationAgent** — ReAct (Chapter 4): iterative Reason-Act-Observe loop with tools
- **PostmortemAgent** — Reflection (Chapter 4): draft → critique → revise

In [4]:
from src.core.llm_client import HelloAgentsLLM
from src.agents.triage_agent import TriageAgent
from src.agents.investigation_agent import InvestigationAgent
from src.agents.postmortem_agent import PostmortemAgent
from src.agents.pipeline import run_pipeline, list_incidents, load_incident

# Verify LLM connection
llm = HelloAgentsLLM(verbose=True)
test_response = llm.think([{"role": "user", "content": "Reply with exactly: LLM OK"}])
print(f"\nLLM connection test: {test_response.strip()}")

print(f"\nAvailable incident fixtures: {list_incidents()}")

🧠 Calling deepseek-v4-flash...
LLM OK

LLM connection test: LLM OK

Available incident fixtures: ['db_pool_exhaustion', 'external_api_ratelimit', 'memory_leak_oom']


---
## Part 5: Demo — Full Pipeline

### Demo 1: DB Connection Pool Exhaustion (INC-001)

**Alert:** `checkout-service` P99 latency 8.3s > threshold 1.0s  
**Root cause (ground truth):** Missing index on `orders.user_id` → full table scan → DB pool exhaustion

In [5]:
# Show the incident alert
incident_1 = load_incident("db_pool_exhaustion")
print("📣 Incident Alert:")
print(json.dumps({
    "incident_id": incident_1["incident_id"],
    "service": incident_1["service"],
    "severity": incident_1["severity"],
    "alert": incident_1["alert"]
}, indent=2))

📣 Incident Alert:
{
  "incident_id": "INC-001",
  "service": "checkout-service",
  "severity": "P1",
  "alert": {
    "metric": "http_request_duration_p99",
    "value": 8.3,
    "threshold": 1.0,
    "unit": "seconds",
    "timestamp": "2024-01-15T14:03:22Z",
    "description": "P99 request latency critically exceeds SLO threshold"
  }
}


In [6]:
# Run the full pipeline
start = time.time()
result_1 = run_pipeline("db_pool_exhaustion", verbose=True)
elapsed_1 = round(time.time() - start, 1)
print(f"\n⏱️  Pipeline completed in {elapsed_1}s")


🚨 STAGE 1: TRIAGE — Generating investigation plan
🧠 Calling deepseek-v4-flash...
```python
[
  {"tool": "metric_query", "query": "http_request_duration_p99{service=\"checkout-service\"}[10m]", "reason": "Confirm latency spike and observe time range"},
  {"tool": "log_search", "query": "service=checkout-service AND level=ERROR", "reason": "Surface errors correlated with the p99 spike"},
  {"tool": "metric_query", "query": "sum by (status) (rate(http_requests_total{service=\"checkout-service\"}[5m]))", "reason": "Check for a corresponding error-rate spike and 5xx statuses"},
  {"tool": "log_search", "query": "service=checkout-service AND (timeout|deadline|upstream)", "reason": "Look for upstream timeouts or dependency failures"},
  {"tool": "metric_query", "query": "http_request_duration_p99{service=\"checkout-service\", dependency=\"payment-service\"}[5m]", "reason": "Determine if slowdown is from the payment dependency"},
  {"tool": "runbook_lookup", "query": "checkout-service high la

In [7]:
# Display the generated post-mortem report
from IPython.display import Markdown, display
display(Markdown(result_1["report"]))

# Post-Mortem: INC-001 — Checkout Service P1 Latency Incident

## Executive Summary

On 2024-01-15 at 14:00Z, checkout-service was operating normally. At 14:01Z, `http_request_duration_p99` crossed the 1.0s SLO and continued to escalate. By 14:03Z, P99 latency peaked at **8.3s**, affecting **2,847 users**. A P1 alert fired at **14:03:22Z**.

The direct cause was database-side degradation: a missing index on `orders(user_id)` caused full table scans on a 2.1M-row table, `db_query_duration_p99` reached **30s**, and the database connection pool was exhausted. Incident mitigation required terminating slow queries, hot-increasing the DB pool from 10 to 20 connections, and creating `idx_orders_user_id`.

The systemic root cause was not just the missing index. The query pattern shipped without a database performance review, CI had no query-plan guardrail, no database query duration alert existed, and the latency alert’s 2-minute `for` clause delayed detection until 14:03:22Z. Recovery was confirmed at **14:08:20Z**, giving an incident duration of **7 minutes 20 seconds**.

## Incident Timeline

All times are UTC.

| Time | Event |
|---|---|
| 14:00:00Z | Baseline normal: `http_request_duration_p99=0.3s`, `request_queue_depth=12`, `db_query_duration_p99=45ms`. |
| 14:01:00Z | Degradation begins: `http_request_duration_p99=1.2s` — crossed SLO; `request_queue_depth=85`; `db_query_duration_p99=280ms`. |
| 14:02:00Z | Escalation: `http_request_duration_p99=4.7s`; `request_queue_depth=220`; `db_query_duration_p99=12s`. |
| 14:03:00Z | Peak impact: `http_request_duration_p99=8.3s`; `request_queue_depth=450`; `db_query_duration_p99=30s`. |
| 14:03:22Z | **P1 alert fired**: `http_request_duration_p99` = 8.3s > 1.0s threshold. The alert rule required the breach to persist for 2 minutes; the first qualified evaluation after the pending period was 14:03:22Z. This was not a threshold misconfiguration, but the `for` clause was too long for a checkout-critical path. |
| 14:03:25Z | On-call engineer acknowledged the page. |
| 14:03:40Z | Runbook `checkout-service v1.2` consulted; pattern **"DB pool exhausted"** matched. No ERROR logs and no payment dependency failure were observed. |
| 14:04:15Z | `SHOW PROCESSLIST` run on primary DB; found 11 queries running longer than 5 seconds, longest 18 seconds, mostly `SELECT ... FROM orders WHERE user_id = ?`. |
| 14:04:40Z | `EXPLAIN` on the slowest query confirmed full table scan: `type=ALL`, `rows=2,100,000`, no usable index on `orders(user_id)`. |
| 14:05:00Z | All 11 slow queries terminated with `KILL QUERY <id>`. |
| 14:05:20Z | DB pool size hot-reloaded from 10 to 20 via config hot-reload. |
| 14:06:15Z | Created index: `CREATE INDEX idx_orders_user_id ON orders(user_id);`. |
| 14:06:35Z | `EXPLAIN` verified index usage: `type=ref`, `key=idx_orders_user_id`, `rows=18`. |
| 14:07:00Z | `kubectl rollout restart deployment/checkout-service` issued. |
| 14:07:45Z | Rollout complete; pods ready. |
| 14:08:20Z | Metrics confirmed below SLO for 3 consecutive checks: `http_request_duration_p99=0.52s`, `request_queue_depth=18`, `db_query_duration_p99=64ms`. Incident declared resolved. |

**Detection and response times:**

- SLO breach to alert: **2 minutes 22 seconds** (14:01:00Z → 14:03:22Z)
- Alert to first remediation action: **53 seconds** (14:03:22Z → 14:04:15Z)
- Alert to slow-query termination: **1 minute 38 seconds** (14:03:22Z → 14:05:00Z)
- Alert to index creation: **2 minutes 53 seconds** (14:03:22Z → 14:06:15Z)
- Alert to recovery confirmed: **4 minutes 58 seconds** (14:03:22Z → 14:08:20Z)

## Root Cause Analysis

### Direct Cause

The `orders` table had no index on `user_id`. A newly introduced query pattern filtered orders by `user_id`, forcing a full table scan of ~2.1M rows per request. This caused individual query latencies up to 30 seconds, which exhausted the 10-connection database pool and caused upstream request queueing.

### Expanded 5 Whys

1. **Why did checkout P99 latency reach 8.3s?**  
   Incoming requests piled up in the service queue; `request_queue_depth` reached 450.

2. **Why did the queue pile up?**  
   The database connection pool was exhausted — all connections were busy and unavailable.

3. **Why were database connections unavailable?**  
   Queries were extremely slow; `db_query_duration_p99` reached 30s, so each connection remained occupied for far too long.

4. **Why were queries so slow?**  
   Queries on `orders` performed full table scans because there was no index on `user_id`.

5. **Why was the index missing?**  
   The query pattern was introduced without a database performance review and without a schema migration creating the needed index.

6. **Why could it ship without a review/guardrail?**  
   The deployment checklist did not require DBA or query-plan review for new query patterns, CI did not run `EXPLAIN` on SQL queries or migrations, the performance regression suite did not include query-plan analysis, and no alert existed for database query duration. The only safeguard was the user-facing latency SLO alert, and its 2-minute `for` clause delayed detection.

### Systemic Root Cause

Checkout-service delivery lacked automated database performance guardrails. There was no CI check to catch full table scans, no mandatory migration approval gate, and no direct alert on database query latency or connection pool exhaustion. This allowed a single missing index to cascade into a public-facing P1 incident.

## Impact Assessment

- **Severity:** P1 — public-facing checkout service with critical SLO violation.
- **Affected users:** 2,847.
- **Incident duration:** 7 minutes 20 seconds (14:01:00Z → 14:08:20Z).
- **Business impact:** Users experienced checkout latency up to 8.3 seconds. At queue depth 450, many requests likely timed out or were abandoned, causing lost orders and revenue impact.
- **Technical impact:** Database query latency at P99 reached 30 seconds; DB connection pool was exhausted; request queue depth reached 450. No ERROR logs were emitted, and no payment dependency failure was observed, which isolated the issue to the primary database layer.

## Remediation Actions Taken

The following actions were executed during the incident:

1. Ran `SHOW PROCESSLIST` on the primary DB and identified 11 queries longer than 5 seconds.
2. Ran `EXPLAIN` on the slowest query and confirmed a full table scan on `orders`.
3. Terminated the slow queries with `KILL QUERY <id>`.
4. Hot-increased the DB pool size from 10 to 20.
5. Created the missing index:
   ```sql
   CREATE INDEX idx_orders_user_id ON orders(user_id);
   ```
6. Ran `EXPLAIN` again and confirmed index usage.
7. Restarted service pods:
   ```bash
   kubectl rollout restart deployment/checkout-service
   ```
8. Monitored `db_pool_active_connections`, `request_queue_depth`, `db_query_duration_p99_ms`, and `http_request_duration_p99` until stable and below SLO.

## Action Items

| Action Item | Owner | Due Date | Priority | Success Criteria |
|---|---|---|---|---|
| Persist and verify `idx_orders_user_id` in migration scripts and all environments | DBA | 2024-01-15 | P0 | Merge migration `003_add_idx_orders_user_id.sql`; `SHOW INDEX FROM orders` shows the index in prod and staging; attach output to INC-001. |
| Add database query duration and connection pool alerts | SRE | 2024-01-16 | P1 | Prometheus alerts exist for `db_query_duration_p99 > 1s` for 1m and connection pool saturation > 80% for 2m; test alerts fire in staging. |
| Enable slow query logging on the primary database | DBA | 2024-01-16 | P1 | Slow query log is enabled with threshold 1s and is querying an alerting pipeline. |
| Reduce latency alert delay | SRE | 2024-01-16 | P1 | Change `CheckoutServiceP99Latency` `for` from 2m to 1m and evaluation interval to 15s; synthetic alert fires within 90s of threshold breach. |
| Add CI query-plan check for new SQL and migrations | Checkout Team + DBA | 2024-01-17 | P1 | CI runs `EXPLAIN` on every new SQL query/migration; fails if a full table scan or missing index is detected; verified with a deliberately failing test query. |
| Make DBA review mandatory for DB-affecting changes | Checkout Team | 2024-01-17 | P1 | GitHub CODEOWNERS requires DBA approval for all migration and query-plan-affecting changes; no change merges without approval. |
| Update checkout-service runbook | SRE | 2024-01-16 | P2 | Runbook v1.3 includes exact `SHOW PROCESSLIST`, `EXPLAIN`, `KILL QUERY`, DB pool hot-reload, and post-recovery verification commands. |
| Load-test DB pool and autoscaling settings | SRE | 2024-01-18 | P2 | Load test at 2x peak shows the pool is not exhausted; connection timeout and HPA settings are documented and linked to INC-001. |

## Lessons Learned

- A single missing index cascaded into full table scans → slow DB queries → DB pool exhaustion → user-facing latency. We are not relying on a one-time fix. The CI query-plan check and DBA approval gate in the Action Items will catch this class of issue before production.
- No ERROR logs were emitted during the incident, which made detection dependent on latency SLO alerts. We are adding direct alerts on `db_query_duration_p99` and connection pool saturation so future database degradation is detected before user-facing SLOs are breached.
- The latency alert’s 2-minute `for` clause delayed the page by 2 minutes 22 seconds. For a checkout-critical path, this is too slow. The alerting behavior change in the Action Items reduces that delay to under 90 seconds.
- Increasing the DB pool size bought time, but the permanent fix was removing the slow query by adding the missing index. The runbook now clearly separates temporary mitigations from permanent fixes and includes evidence-gathering steps.
- The runbook pattern **"DB pool exhausted"** matched exactly and provided clear remediation steps. Keeping runbooks precise shortened the response time. We are updating the runbook with the exact commands and timestamps from this incident to make future responses even faster.

### Demo 2: External API Rate Limit Cascade (INC-003)

**Alert:** `payment-service` success rate 23.4% < threshold 95%  
**Root cause (ground truth):** Promotional traffic spike + no exponential backoff → retry storm → Stripe 429 cascade

In [8]:
start = time.time()
result_3 = run_pipeline("external_api_ratelimit", verbose=True)
elapsed_3 = round(time.time() - start, 1)
print(f"\n⏱️  Pipeline completed in {elapsed_3}s")
display(Markdown(result_3["report"]))


🚨 STAGE 1: TRIAGE — Generating investigation plan
🧠 Calling deepseek-v4-flash...
```python
[
  {"tool": "metric_query", "query": "payment_success_rate{service=\"payment-service\"}[30m]", "reason": "Confirm the success rate drop time window and trend"},
  {"tool": "metric_query", "query": "payment_failure_rate{service=\"payment-service\"}[30m]", "reason": "Correlate failure rate with success rate drop to isolate failure pattern"},
  {"tool": "log_search", "query": "service=payment-service AND level=ERROR", "reason": "Identify error-level logs around the incident timestamp"},
  {"tool": "log_search", "query": "exception|timeout|connection_refused", "reason": "Find specific exception patterns that explain payment failures"},
  {"tool": "runbook_lookup", "query": "payment service low success rate", "reason": "Check for known remediation steps for payment success rate degradation"}
]
```

✅ Investigation plan (5 steps):
   1. [metric_query] payment_success_rate{service="payment-service"}[3

KeyboardInterrupt: 

---
## Part 6: Performance Evaluation

Run all 3 incident fixtures and measure: pipeline speed, root cause accuracy, and report quality.

In [ ]:
GROUND_TRUTH = {
    "db_pool_exhaustion":   "missing index",
    "memory_leak_oom":      "no TTL",
    "external_api_ratelimit": "retry storm",
}

eval_results = []
for incident_id in list_incidents():
    t0 = time.time()
    res = run_pipeline(incident_id, verbose=False)
    elapsed = round(time.time() - t0, 1)

    ground_truth_keyword = GROUND_TRUTH.get(incident_id, "")
    root_cause_text = res["findings"].get("root_cause", "").lower()
    correct = ground_truth_keyword.lower() in root_cause_text

    eval_results.append({
        "incident_id": incident_id,
        "service": res["service"],
        "severity": res["severity"],
        "elapsed_s": elapsed,
        "root_cause_correct": "✅" if correct else "❌",
        "root_cause_found": res["findings"].get("root_cause", "")[:80] + "...",
    })

print("\n=== Evaluation Results ===")
for r in eval_results:
    print(f"  [{r['root_cause_correct']}] {r['incident_id']:30s} "
          f"| {r['elapsed_s']:5.1f}s | {r['root_cause_found']}")

accuracy = sum(1 for r in eval_results if r["root_cause_correct"] == "✅") / len(eval_results)
avg_time = sum(r["elapsed_s"] for r in eval_results) / len(eval_results)
print(f"\nAccuracy: {accuracy:.0%}   Avg pipeline time: {avg_time:.1f}s")

---
## Part 7: Summary and Outlook

### What We Built
- **TriageAgent (Plan-and-Solve)**: converts a raw alert into an ordered investigation plan, reducing hallucinated tool calls in the ReAct stage
- **InvestigationAgent (ReAct)**: iterates through log search → metric queries → runbook lookup to arrive at a root cause with evidence
- **PostmortemAgent (Reflection)**: produces a structured RCA report and self-critiques it against quality criteria before finalizing
- **FastAPI backend**: exposes the pipeline as a REST API, ready for frontend integration

### Agent Paradigms Demonstrated
| Agent | Paradigm | Chapter Reference |
|---|---|---|
| TriageAgent | Plan-and-Solve | Chapter 4 |
| InvestigationAgent | ReAct | Chapter 4 |
| PostmortemAgent | Reflection | Chapter 4 |

### Challenges and Lessons
1. **Structured output parsing**: LLMs don't always return valid JSON/Python lists. Robust parsing with fallback logic is essential.
2. **Tool design**: Tool descriptions must be precise — ambiguous descriptions cause the ReAct agent to call the wrong tool.
3. **Context window management**: Passing the full investigation history in each ReAct turn grows quickly. In production, use a sliding window or summarization.
4. **Reflection threshold**: The Reflection pattern only adds value if the critique is rigorous. Vague critique prompts produce vague improvements.

### Future Plans
- [ ] **SSE streaming**: stream agent reasoning steps to a frontend in real-time
- [ ] **Vue/React frontend**: incident selector UI + live reasoning trace + markdown report viewer
- [ ] **Real log ingestion**: connect to actual log aggregators (Loki, CloudWatch, Datadog)
- [ ] **Memory across incidents**: vector store for past RCA reports to accelerate future investigations
- [ ] **Automated runbook execution**: give the agent permission to run safe remediation commands (kubectl, SQL index creation)